# Feature attributions

Plot some feature attributions and find lower and upper bound for random baseline

In [ ]:
from matplotlib import pyplot as plt
import torch
import numpy as np

from captum.attr import configure_interpretable_embedding_layer
from captum.attr import remove_interpretable_embedding_layer
from captum.attr import InputXGradient
from captum.attr import IntegratedGradients
from captum.attr import KernelShap

import config
import loader

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
N_STEPS = 50
N_SAMPLES = 50

## LAAT

In [ ]:
test_data, vocab, args_new = loader.load_laat_data(config.LAAT_ARGS)
dataloader = loader.create_laat_dataloader(test_data, vocab, args_new)
model = loader.load_laat(vocab, args_new)
def model_wrapper(*args, **kwargs):
    output, attn_weights = model(*args, **kwargs)
    return torch.sigmoid(output[1])

In [ ]:
# Find most frequent labels
freq_dict = {}
for idx, tup in enumerate(dataloader):
    _, labels, _, _ = tup
    labels = labels[1]
    non_zero_labels = torch.nonzero(labels[0], as_tuple=False).flatten().tolist()
    for label in non_zero_labels:
        freq_dict[label] = freq_dict[label] + 1 if label in freq_dict else 1
# Sort labels descending by frequency
labels_sorted = sorted(freq_dict, key=freq_dict.get, reverse=True)

In [ ]:
int_emb = configure_interpretable_embedding_layer(model, 'embedding')
model.train()
attributors = {
    'ixg': InputXGradient(model_wrapper),
    'ig': IntegratedGradients(model_wrapper),
    'shap': KernelShap(model_wrapper)
}

In [ ]:
# Choose random subset of test samples
sub_bits = np.array([0] * (config.N_TEST - config.N_SUB) + [1] * (config.N_SUB))
np.random.seed(config.SEED)
np.random.shuffle(sub_bits)

In [ ]:
predss = {}
attrs = {}
for idx, tup in enumerate(dataloader):
    # Attribute only the first five samples of the subset
    if sub_bits[idx] == 0 or np.count_nonzero(sub_bits[:idx]) > 5:
        continue
    print("Attributing sample", idx)

    predss[idx] = {}
    attrs[idx] = {}

    # Prepare input and baseline
    input_indices, _, afa, _ = tup
    input_indices = input_indices.to(DEVICE)
    input_embed = int_emb.indices_to_embeddings(input_indices).to(DEVICE)
    base_embed = torch.zeros_like(input_embed).to(DEVICE)

    preds = model_wrapper(input_embed, afa)[0]

    # Attribute sample
    for target_idx, pred in enumerate(preds):
        # Attribute only positive predictions of the five most frequent labels
        if pred.item() > config.THRESHOLD and target_idx in labels_sorted[:5]:
            print("Attributing target_idx", target_idx)
            predss[idx][target_idx] = pred.item()
            a = {}
            # Compute attributions
            a['ixg'] = attributors['ixg'].attribute(input_embed, \
                            additional_forward_args = afa, \
                            target = target_idx).float()
            a['ig'] = attributors['ig'].attribute(input_embed, \
                            base_embed, \
                            internal_batch_size = config.INT_BATCH, \
                            additional_forward_args = afa, \
                            target = target_idx, \
                            n_steps = N_STEPS).float()
            # For some reason, KernelShap needs afa in different shape
            afa = afa.unsqueeze(0)
            with torch.no_grad():
                a['shap'] = attributors['shap'].attribute(input_embed, \
                            target = target_idx, \
                            n_samples = N_SAMPLES, \
                            additional_forward_args = afa).float()
            afa = afa.squeeze(0)
            attrs[idx][target_idx] = a

In [ ]:
# Find lower and upper bound for random attributions
lb, ub = 0, 0
for idx, preds in predss.items():
    for target_idx, pred in preds.items():
        plb = min([torch.min(attrs[idx][target_idx][method_name]).item() for method_name in ['ixg', 'ig', 'shap']])
        pub = max([torch.max(attrs[idx][target_idx][method_name]).item() for method_name in ['ixg', 'ig', 'shap']])
        lb = plb if plb < lb else lb
        ub = pub if pub > ub else ub
print("Lower bound:", lb)
print("Upper bound:", ub)

In [ ]:
# Plot attributions
LB, UB = -0.1, 0.1
for idx, preds in predss.items():
    for target_idx, pred in preds.items():
        print("sample_idx:", idx)
        print("target_idx:", target_idx)
        print("pred:", round(pred, 2))
        attrs_ixg = attrs[idx][target_idx]['ixg'].sum(dim=2).squeeze(0).cpu().detach().numpy()
        attrs_ig = attrs[idx][target_idx]['ig'].sum(dim=2).squeeze(0).cpu().detach().numpy()
        attrs_shap = attrs[idx][target_idx]['shap'].sum(dim=2).squeeze(0).cpu().detach().numpy()
        attrs_ra = (UB - LB) * torch.rand_like(attrs[idx][target_idx]['ixg']) + LB
        attrs_ra = attrs_ra.sum(dim=2).squeeze(0).cpu().detach().numpy()
        print('attrs_ixg plot:')
        plt.bar(range(len(attrs_ixg)), attrs_ixg)
        plt.show()
        print('attrs_ig plot:')
        plt.bar(range(len(attrs_ig)), attrs_ig)
        plt.show()
        print('attrs_shap plot:')
        plt.bar(range(len(attrs_shap)), attrs_shap)
        plt.show()
        print('attrs_ra plot:')
        plt.bar(range(len(attrs_ra)), attrs_ra)
        plt.show()

In [ ]:
model.train(mode=False)
remove_interpretable_embedding_layer(model, int_emb)

## CAML

In [ ]:
model, args_new, dicts = loader.load_caml(config.CAML_ARGS)
dataloader = loader.create_caml_dataloader(args_new, dicts)
# Convert caml dataloader to list because generator can't be iterated twice
dataloader = list(dataloader)
def model_wrapper(*args, **kwargs):
    output, loss, alpha = model(*args, **kwargs)
    return torch.sigmoid(output)

In [ ]:
# Find most frequent labels
freq_dict = {}
for idx, tup in enumerate(dataloader):
    _, labels, _, _, _ = tup
    labels = torch.FloatTensor(labels)
    non_zero_labels = torch.nonzero(labels[0], as_tuple=False).flatten().tolist()
    for label in non_zero_labels:
        freq_dict[label] = freq_dict[label] + 1 if label in freq_dict else 1
# Sort labels descending by frequency
labels_sorted = sorted(freq_dict, key=freq_dict.get, reverse=True)

In [ ]:
int_emb = configure_interpretable_embedding_layer(model, 'embed')
model.train()
attributors = {
    'ixg': InputXGradient(model_wrapper),
    'ig': IntegratedGradients(model_wrapper),
    'shap': KernelShap(model_wrapper)
}

In [ ]:
# Choose random subset of test samples
sub_bits = np.array([0] * (config.N_TEST - config.N_SUB) + [1] * (config.N_SUB))
np.random.seed(config.SEED)
np.random.shuffle(sub_bits)

In [ ]:
predss = {}
attrs = {}
for idx, tup in enumerate(dataloader):
    # Attribute only the first five samples of the subset
    if sub_bits[idx] == 0 or np.count_nonzero(sub_bits[:idx]) > 5:
        continue
    print("Attributing sample", idx)

    predss[idx] = {}
    attrs[idx] = {}

    # Prepare input and baseline
    input_indices, afa, _, _, _ = tup
    input_indices = torch.LongTensor(input_indices).to(DEVICE)
    afa = torch.FloatTensor(afa).to(DEVICE)
    input_embed = int_emb.indices_to_embeddings(input_indices).to(DEVICE)
    base_embed = torch.zeros_like(input_embed).to(DEVICE)

    preds = model_wrapper(input_embed, afa)[0]

    # Attribute sample
    for target_idx, pred in enumerate(preds):
        # Attribute only positive predictions of the five most frequent labels
        if pred.item() > config.THRESHOLD and target_idx in labels_sorted[:5]:
            print("Attributing target_idx", target_idx)
            predss[idx][target_idx] = pred.item()
            a = {}
            # Compute attributions
            a['ixg'] = attributors['ixg'].attribute(input_embed, \
                            additional_forward_args = afa, \
                            target = target_idx).float()
            a['ig'] = attributors['ig'].attribute(input_embed, \
                            base_embed, \
                            internal_batch_size = config.INT_BATCH, \
                            additional_forward_args = afa, \
                            target = target_idx, \
                            n_steps = N_STEPS).float()
            # For some reason, KernelShap needs afa in different shape
            afa = afa.unsqueeze(0)
            with torch.no_grad():
                a['shap'] = attributors['shap'].attribute(input_embed, \
                            target = target_idx, \
                            n_samples = N_SAMPLES, \
                            additional_forward_args = afa).float()
            afa = afa.squeeze(0)
            attrs[idx][target_idx] = a

In [ ]:
# Find lower and upper bound for random attributions
lb, ub = 0, 0
for idx, preds in predss.items():
    for target_idx, pred in preds.items():
        plb = min([torch.min(attrs[idx][target_idx][method_name]).item() for method_name in ['ixg', 'ig', 'shap']])
        pub = max([torch.max(attrs[idx][target_idx][method_name]).item() for method_name in ['ixg', 'ig', 'shap']])
        lb = plb if plb < lb else lb
        ub = pub if pub > ub else ub
print("Lower bound:", lb)
print("Upper bound:", ub)

In [ ]:
# Plot attributions
LB, UB = -0.1, 0.1
for idx, preds in predss.items():
    for target_idx, pred in preds.items():
        print("sample_idx:", idx)
        print("target_idx:", target_idx)
        print("pred:", round(pred, 2))
        attrs_ixg = attrs[idx][target_idx]['ixg'].sum(dim=2).squeeze(0).cpu().detach().numpy()
        attrs_ig = attrs[idx][target_idx]['ig'].sum(dim=2).squeeze(0).cpu().detach().numpy()
        attrs_shap = attrs[idx][target_idx]['shap'].sum(dim=2).squeeze(0).cpu().detach().numpy()
        attrs_ra = (UB - LB) * torch.rand_like(attrs[idx][target_idx]['ixg']) + LB
        attrs_ra = attrs_ra.sum(dim=2).squeeze(0).cpu().detach().numpy()
        print('attrs_ixg plot:')
        plt.bar(range(len(attrs_ixg)), attrs_ixg)
        plt.show()
        print('attrs_ig plot:')
        plt.bar(range(len(attrs_ig)), attrs_ig)
        plt.show()
        print('attrs_shap plot:')
        plt.bar(range(len(attrs_shap)), attrs_shap)
        plt.show()
        print('attrs_ra plot:')
        plt.bar(range(len(attrs_ra)), attrs_ra)
        plt.show()

In [ ]:
model.train(mode=False)
remove_interpretable_embedding_layer(model, int_emb)